# 01 · Sampling + PaddleOCR on GPU

Runs the slow, GPU-bound part of the pipeline: download 4,000 no-photo CV images and run
PaddleOCR on them, then the cheap CPU steps (OCR quality, cleaning, split) while the images
are here. **Keep PaddlePaddle in this notebook only** — PyTorch work goes in `02_models_gpu.ipynb`
(Paddle and PyTorch in one runtime often clash over CUDA/cuDNN versions).

**Before running**
* **Kaggle:** *Settings → Accelerator → GPU T4 x2* (or P100), *Internet → On*,
  *Add-ons → Secrets →* `HF_TOKEN`. For the full run use **Save Version → Save & Run All** so it keeps
  running with the browser closed (max 12 h per session).
* **Colab:** *Runtime → Change runtime type → T4 GPU*; key icon (Secrets) → `HF_TOKEN`, with notebook access on.
  Drive is mounted so OCR results survive a disconnect: just re-run all cells and it resumes.
* On huggingface.co, accept the access conditions of `sumin-yu/PopResume` with the same account as the token.


In [ ]:
# ---- repository and working branch ----
REPO_URL = "https://github.com/karelysscs/cv_experience_ocr_bert.git"
BRANCH = "Practica_04"

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")
BASE = "/kaggle/working" if IN_KAGGLE else "/content"
REPO = f"{BASE}/cv_experience_ocr_bert"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
subprocess.run(["git", "pull"], check=False)
print("Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "other", "| cwd:", os.getcwd())
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1 · Install PaddlePaddle (GPU) + PaddleOCR
The `cu126` index matches CUDA 12.x; if `nvidia-smi` shows another CUDA, pick the matching index from paddlepaddle.org.cn.

In [ ]:
!pip install -q paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q paddleocr==3.7.0 jiwer==4.0.0 datasets huggingface_hub
import paddle
paddle.utils.run_check()
print("paddle", paddle.__version__, "| CUDA build:", paddle.device.is_compiled_with_cuda(),
      "| device:", paddle.device.get_device())

## 2 · Persist data across disconnects (Colab only)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = "/content/drive/MyDrive/cv_experience_ocr_bert_cache"
    for sub in ["data/raw", "data/ocr"]:
        target = f"{CACHE}/{sub}"
        os.makedirs(target, exist_ok=True)
        if not os.path.islink(sub):
            subprocess.run(["rm", "-rf", sub]); os.makedirs(os.path.dirname(sub), exist_ok=True)
            os.symlink(target, sub)
    os.makedirs("data/raw/images", exist_ok=True)
    print("data/raw and data/ocr live in", CACHE)

## 3 · Hugging Face login

In [ ]:
# Hugging Face token, stored as a secret named HF_TOKEN (never paste it in the code)
if IN_COLAB:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
elif IN_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
else:
    token = os.environ["HF_TOKEN"]
os.environ["HF_TOKEN"] = token
from huggingface_hub import login, whoami
login(token=token)
print("Logged in as", whoami()["name"])

## 4 · Stratified sample (800 per occupation, seed 42)
Streams one occupation at a time; already-saved images are not written again.

In [ ]:
!python -m src.download_data --n-per-occupation 800

## 5 · PaddleOCR
First a 5-CV smoke test to check speed, then the full run. It skips CVs that already have a JSON, so it can be re-run safely.

In [ ]:
!python -m src.ocr_extraction --model server --device gpu --limit 5
!cat output/metrics/ocr_summary.json

In [ ]:
!python -m src.ocr_extraction --model server --device gpu

## 6 · OCR quality, cleaning, masking, split (CPU, fast)
Read `output/tables/inspection_sample.txt` (50 random CVs) **before** trusting the masking regexes — add patterns in `src/preprocessing.py` if you see other ways of stating experience or age.

In [ ]:
!python -m src.ocr_quality
!python -m src.preprocessing --inspect 50
!head -c 4000 output/tables/inspection_sample.txt

In [ ]:
!python -m src.preprocessing --min-conf 0.80
!python -m src.split
!python -m src.visualization

## 7 · Download the results
The zip holds everything the next steps need **except** the images (only 5 examples for the app, which are git-ignored). Unzip it in your local repo, commit `data/ocr`, `data/processed`, `data/splits`, `data/raw/*.csv|txt` and `output/` on a feature branch, push, and open a PR into `Practica_04`.

In [ ]:
import glob, shutil, zipfile
examples = sorted(glob.glob("data/raw/images/*.png"))[:5]
with zipfile.ZipFile(f"{BASE}/stage1_ocr_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for pattern in ["data/raw/*.csv", "data/raw/*.txt", "data/ocr/*.json", "data/processed/*",
                    "data/splits/*", "output/**/*"]:
        for f in glob.glob(pattern, recursive=True):
            if os.path.isfile(f):
                z.write(f)
    for f in examples:
        z.write(f)
!pip freeze | grep -i -E "paddle|jiwer|datasets" > output/metrics/env_ocr.txt
print("Created", f"{BASE}/stage1_ocr_results.zip", os.path.getsize(f"{BASE}/stage1_ocr_results.zip") // 1e6, "MB")
if IN_COLAB:
    from google.colab import files
    files.download(f"{BASE}/stage1_ocr_results.zip")
# On Kaggle: download it from the notebook's Output tab.